# RQ1: Tabular Synthesizer Benchmark & Rankings

**Research Question 1 (RQ1):** *Which generator achieves the best balanced fidelity, utility, and privacy across benchmark datasets?*

This notebook evaluates tabular synthesizers across standard benchmark datasets (Adult, Credit, Diabetes) and CC18 meta-benchmark datasets using the fail-closed Synthetic Data Trust Lab decision engine.

Key outputs:
- Per-dataset aggregated balanced score rankings
- Statistical significance testing via `autorank` (Friedman test + critical difference CD diagrams)
- Winner vs dataset characteristics analysis

Run headless (analysis env):
`bash scripts/envrun.sh analysis python -m jupyter nbconvert --to notebook --execute --output-dir outputs/notebooks --output rq1_benchmark_executed.ipynb notebooks/rq1_benchmark.ipynb`

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "packages" / "analysis").exists())
STORE_ROOT = os.environ.get("STL_STORAGE_ROOT") or str(REPO / "outputs" / "store")
STORE_INTEGRATED = str(REPO / "outputs" / "store_integrated")
REPORTS_DIR = REPO / "reports"

print(f"Repo:              {REPO}")
print(f"Store root:        {STORE_ROOT}")
print(f"Store integrated:  {STORE_INTEGRATED}")
print(f"Reports directory: {REPORTS_DIR}")

## 1. Load Collected Results

Collects flattened results across stores, calculating fail-closed privacy gates and balanced scores.

In [ ]:
from stl_analysis.collect import collect

stores = [STORE_ROOT]
if Path(STORE_INTEGRATED).exists():
    stores.append(STORE_INTEGRATED)

df = collect(stores, include_meta=True)
print(f"Collected {len(df)} jobs from {df['dataset_id'].nunique()} datasets across {df['run_id'].nunique()} runs.")
display(df[['run_id', 'dataset_id', 'generator', 'state', 'passed_gate', 'balanced_score']].head(10))

## 2. Core Benchmark Rankings & CD Diagram

In [ ]:
ranks_file = REPORTS_DIR / "tables" / "rq1_core_ranks.md"
if ranks_file.exists():
    display(Markdown(ranks_file.read_text(encoding="utf-8")))
else:
    insufficient_file = REPORTS_DIR / "tables" / "rq1_core_insufficient.md"
    if insufficient_file.exists():
        display(Markdown(insufficient_file.read_text(encoding="utf-8")))
    else:
        print("No pre-generated core rank table found. Run make_report.sh to generate.")

cd_png = REPORTS_DIR / "figures" / "rq1_core_cd.png"
if cd_png.exists():
    display(Image(filename=str(cd_png)))
else:
    print("No core CD diagram generated yet (requires >= 5 datasets with common generators).")

## 3. CC18 Meta-Benchmark Panel (25 Datasets)

In [ ]:
cc18_ranks = REPORTS_DIR / "tables" / "rq1_cc18_ranks.md"
if cc18_ranks.exists():
    display(Markdown(cc18_ranks.read_text(encoding="utf-8")))
else:
    print("No CC18 rank table found.")

cc18_cd = REPORTS_DIR / "figures" / "rq1_cc18_cd.png"
if cc18_cd.exists():
    display(Image(filename=str(cc18_cd)))